In [1]:
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error
import numpy as np

In [2]:
MV_users = pd.read_csv('data/users.csv')
MV_movies = pd.read_csv('data/movies.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [3]:
from collections import namedtuple
Data = namedtuple('Data', ['users','movies','train','test'])
data = Data(MV_users, MV_movies, train, test)

In [4]:
# Following module 3 assignment style
class NMFRecommender:
    def __init__(self, data, n_components=20, random_state=42):
        self.data = data
        self.n_components = n_components
        self.random_state = random_state
        self.uid2idx = dict(zip(data.users.uID, range(len(data.users))))
        self.mid2idx = dict(zip(data.movies.mID, range(len(data.movies))))
        self.Mr = self.build_rating_matrix()

    def build_rating_matrix(self):
        n_users = len(self.data.users)
        n_movies = len(self.data.movies)
        Mr = np.zeros((n_users, n_movies))
        for _, row in self.data.train.iterrows():
            uidx = self.uid2idx[row.uID]
            midx = self.mid2idx[row.mID]
            Mr[uidx, midx] = row.rating
        return Mr

    def fit_predict(self):
        model = NMF(n_components=self.n_components, init='random', random_state=self.random_state, max_iter=200)
        W = model.fit_transform(self.Mr)    # Users x Latent Factors
        H = model.components_               # Latent Factors x Movies
        self.pred_matrix = np.dot(W, H)     # Reconstructed Ratings Matrix

    def predict_test_ratings(self):
        preds = []
        for _, row in self.data.test.iterrows():
            uidx = self.uid2idx.get(row.uID)
            midx = self.mid2idx.get(row.mID)
            if uidx is not None and midx is not None:
                preds.append(self.pred_matrix[uidx, midx])
            else:
                preds.append(3)  # Fallback for unseen users/movies
        return np.array(preds)

    def rmse(self, yp):
        yt = self.data.test.rating.values
        return np.sqrt(mean_squared_error(yt, yp))


In [5]:
nmf_model = NMFRecommender(data, n_components=100)  
nmf_model.fit_predict()

yp_nmf = nmf_model.predict_test_ratings()
rmse_nmf = nmf_model.rmse(yp_nmf)

print("NMF RMSE:", rmse_nmf)

c:\Users\Bilal\OneDrive\Documents\Learning\ColoradoMSDS\DTSA 5510\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


NMF RMSE: 3.0195941668302537


Why NMF Performs Worse than cossine and jacsim methods

- Sparsity: NMF doesn’t deal well with very sparse matrices. It assumes all zeros are missing and tries to fill them.

- Global Factorization: It doesn’t use local information (like similar users/movies), unlike similarity-based methods.